# CIFAR-10 CNN Training — Puhti GPU Test
Self-contained notebook: model, data loading, training loop all inline.

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR    = './data'
OUTPUT_DIR  = './output'
EPOCHS      = 10
BATCH_SIZE  = 128
LR          = 0.001
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 10
SEED        = 42
torch.manual_seed(SEED)

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        return self.pool(self.conv(x))

class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64), ConvBlock(64, 128), ConvBlock(128, 256)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256*4*4, 512),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(512, NUM_CLASSES),
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CIFAR10CNN().to(device)
print(f'Device: {device}')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
import os
from torchvision import datasets, transforms

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

_MEAN, _STD = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])
val_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(_MEAN, _STD)])

train_ds = datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=train_tf)
val_ds   = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=val_tf)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=4, pin_memory=True)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
import time, json, sys

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
best_acc = 0.0

print(f'{'Epoch':>6}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}  {'Time':>6}')
print('-'*55)

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    # train
    model.train()
    tr_loss = tr_acc = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()
        tr_acc  += out.argmax(1).eq(y).float().mean().item()
    tr_loss /= len(train_loader); tr_acc /= len(train_loader)
    # val
    model.eval()
    va_loss = va_acc = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            va_loss += criterion(out, y).item()
            va_acc  += out.argmax(1).eq(y).float().mean().item()
    va_loss /= len(val_loader); va_acc /= len(val_loader)
    scheduler.step()

    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({'epoch':epoch,'model':model.state_dict(),'best_acc':best_acc},
                   os.path.join(OUTPUT_DIR,'best_model.pt'))

    print(f'{epoch:>6}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  {va_loss:>8.4f}  {va_acc:>7.4f}  {time.time()-t0:>5.1f}s')
    sys.stdout.flush()

with open(os.path.join(OUTPUT_DIR,'history.json'),'w') as f:
    json.dump(history, f, indent=2)
print(f'\nBest val accuracy: {best_acc:.4f}')

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss'])+1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(epochs, history['train_loss'], label='Train'); ax1.plot(epochs, history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
ax2.plot(epochs, history['train_acc'], label='Train'); ax2.plot(epochs, history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'training_curves.png'), dpi=150)
print(f'Plot saved to {OUTPUT_DIR}/training_curves.png')